# 03 — Clustering the SA2s

**Mini Project 2.** K-Means on the scaled numeric profile, K-Prototypes on the
mixed frame, cluster interpretation, and the comparison against the
administrative geographies that is the contribution of this project.

Two questions to answer by the end:

1. What natural groupings of New Zealand areas exist in census data?
2. How far do those groupings agree with regional council and territorial
   authority — and which explains crowding better?

## Setup

In [ ]:
import sys, json
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mp2

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
np.random.seed(42)

df = pd.read_csv(mp2.OUT / "analysis_frame.csv")
spec = json.loads((mp2.OUT / "feature_spec.json").read_text())

NUMERIC = spec["numeric"]
CATEGORICAL = spec["categorical"]
print(f"{len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical")
print(f"{len(df):,} SA2s to cluster")

## Prepare the clustering matrix

K-Means needs complete, scaled numerics. Two decisions to make explicitly and
defend in the README:

- **Imputation.** Suppressed cells get median-imputed, or the SA2 is dropped.
  Both are defensible; picking one silently is not. Record how many SA2s each
  choice affects.
- **Scaling.** Mandatory. Without it, income in dollars dominates a percentage
  column purely because of its units.

In [ ]:
# TODO: choose drop-or-impute and record the count
# mask = df[NUMERIC].notna().all(axis=1)
# print(f"complete cases: {mask.sum():,} of {len(df):,}")
# cluster_df = df[mask].copy()

In [ ]:
# TODO: scale the numeric matrix for K-Means
# from sklearn.preprocessing import StandardScaler
# X_scaled = StandardScaler().fit_transform(cluster_df[NUMERIC])
# print(X_scaled.shape, X_scaled.mean(axis=0).round(3)[:5])

## Choosing k

Elbow and silhouette, printed as a table. **Do not just take the argmax of the
silhouette** — read the shape. A flat plateau means the data has no strong
cluster structure, and that is a finding worth reporting rather than something
to hide behind a number.

In [ ]:
# diag = mp2.cluster_diagnostics(X_scaled, k_range=range(2, 13))

In [ ]:
# TODO: plot elbow and silhouette side by side
# fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
# axes[0].plot(diag["k"], diag["inertia"], marker="o", color="#2F5D9E")
# axes[0].set_xlabel("k")
# axes[0].set_ylabel("Inertia")
# axes[0].set_title("Elbow: inertia by k")
# axes[1].plot(diag["k"], diag["silhouette"], marker="o", color="#E07B39")
# axes[1].set_xlabel("k")
# axes[1].set_ylabel("Silhouette score")
# axes[1].set_title("Silhouette by k")
# plt.show()

In [ ]:
# TODO: state your k here, in a markdown sentence, with the reason.
K = None  # e.g. 5

## Fit K-Means

In [ ]:
# from sklearn.cluster import KMeans
# km = KMeans(n_clusters=K, n_init=10, random_state=42)
# labels = km.fit_predict(X_scaled)
# cluster_df = mp2.cluster_label_frame(cluster_df, labels, "cluster")
# print(cluster_df["cluster"].value_counts().sort_index())

## TODO — describe each cluster in plain language

This is the interpretation work and it is what the marker reads. For each
cluster, give it a human name and a sentence a non-specialist understands
("outer-suburban mortgaged families on above-median incomes") and the two or
three numbers that justify it.

The profile table below gives you the means to read; do the naming yourself.

In [ ]:
# profile = cluster_df.groupby("cluster")[NUMERIC].mean().round(2)
# counts = cluster_df["cluster"].value_counts().sort_index()
# profile.insert(0, "n_sa2", counts)
# print(profile.T.to_string())

In [ ]:
# TODO: your cluster names go here, keyed by label
CLUSTER_NAMES = {
    # 0: "...",
    # 1: "...",
}
# print({k: CLUSTER_NAMES.get(k) for k in sorted(cluster_df['cluster'].unique())})

## Clusters in geographic space

Your advantage. A cluster solution that only exists in a table is half-argued;
one that visibly occupies a part of the country is convincing.

In [ ]:
# import geopandas as gpd
# gdf = gpd.read_file(mp2.OUT / "analysis_frame.gpkg")
# gdf = gdf.merge(cluster_df[["sa2_code", "cluster"]], on="sa2_code", how="inner")

# fig, ax = plt.subplots(figsize=(8, 9), layout="constrained")
# gdf.plot(column="cluster", categorical=True, cmap="tab10", legend=True,
#          edgecolor="white", linewidth=0.1, ax=ax,
#          legend_kwds={"title": "Cluster", "loc": "lower left", "fontsize": 8})
# ax.set_axis_off()
# ax.set_title("SA2 clusters from census socio-economic profile, 2023")
# plt.show()

In [ ]:
# TODO: cross-tabulate clusters against the administrative geographies
# print(pd.crosstab(cluster_df["cluster"], cluster_df["Regional_council_name"]))

## K-Prototypes — keeping the categoricals

Module 6 material, and better suited to this data than K-Means: K-Prototypes
handles numeric and categorical variables together, so you are not forced to
dummy-encode tenure, composition or urban/rural away. Compare the two
solutions — if they agree, that is evidence the structure is real.

In [ ]:
# TODO: K-Prototypes needs numeric actually numeric and categoricals fine as-is.
# from kmodes.kprototypes import KPrototypes
#
# kp_df = cluster_df[NUMERIC + CATEGORICAL].copy()
# for c in CATEGORICAL:
#     kp_df[c] = kp_df[c].astype(str)
#
# cat_idx = [kp_df.columns.get_loc(c) for c in CATEGORICAL]
# kp = KPrototypes(n_clusters=K, init="Huang", n_init=5, random_state=42)
# kp_labels = kp.fit_predict(kp_df.to_numpy(), categorical=cat_idx)
# print(pd.Series(kp_labels).value_counts().sort_index())
# print(f"final cost: {kp.cost_:.2f}")

In [ ]:
# from sklearn.metrics import adjusted_rand_score
# print("ARI, K-Means vs K-Prototypes:",
#       round(adjusted_rand_score(labels, kp_labels), 3))

## The comparison this project turns on

ARI and AMI between the data-driven clusters and each administrative
geography. Low values mean the drawn boundaries do not correspond to how
areas actually differ — which is the finding, if it holds up.

Read ARI correctly: it is adjusted for chance, so 0 means no better than
random agreement and 1 means identical partitions. A value around 0.2 is
meaningful disagreement, not a failure.

In [ ]:
# TODO: run for each administrative level
# mp2.compare_to_admin(cluster_df, "cluster", "Regional_council_name")
# mp2.compare_to_admin(cluster_df, "cluster", "Territorial_authority_name")
# mp2.compare_to_admin(cluster_df, "cluster", "SA3_name")

## TODO — optional robustness check at SA1

Worth doing if time allows, and almost nobody in the cohort will. SA1s hold
100–200 residents, so suppression is much heavier — but if the cluster
structure holds at both levels you have a robustness claim that is genuinely
yours.

- [ ] Pull the same variables by SA1
- [ ] Repeat the clustering and compare
- [ ] Report the agreement, and the extra suppression, honestly

## Save the labels

Notebook 04 tests whether the cluster label adds predictive power. Write it out
so the two notebooks cannot drift apart.

In [ ]:
# cluster_df[["sa2_code", "cluster"]].to_csv(
#     mp2.OUT / "cluster_labels.csv", index=False
# )
# print("written cluster_labels.csv:", len(cluster_df), "rows")